# DSA211: Diagnosing and Remedying Nonlinearity in Bike-Sharing Demand Models
## Notebook 02: Data Processing & Pipeline Building
**Group 2**: Kok Wai Ying, Lee Sin Yang, Lim Wei Lun Marcus, Samuel Teo Yuzhen, Tee Jing Yuan

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and plotting functions (`*Code: Generated by AI Assistant*`). All modeling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below (see [`agents.md`](../agents.md)).

---

### Objectives & Workflow Compliance (Referencing `docs/project_cv_workflow.html`)
This notebook executes **Steps 1, 3, and 4** of the master cross-validation workflow:
1. **Step 1 (Check Data Quality, Then Split Once)**:
   - Audit data quality (shape, types, missingness markers, duplicates, target identity).
   - Enforce chronological holdout split: Training set (Jan 2011 – Sep 2012) and Test set (Oct – Dec 2012).
   - Lock away the test set into a standalone file (`test_locked.csv`) until the final audit in Notebook 06.
2. **Step 3 (Build a Pipeline)**:
   - Separate data-independent transformations (outside pipeline) from data-dependent transformations that learn parameters from folds (inside `Pipeline` / `ColumnTransformer`).
   - Construct reusable modular preprocessors for baseline, refined, and regularized models to prevent data leakage.
3. **Step 4 (Set Folds and Metric)**:
   - Fix the shared 10-fold cross-validation protocol (`KFold(n_splits=10, shuffle=True, random_state=2026)`).
   - Fix evaluation metrics: count-scale MSE / RMSE and MAE, with fold standard error ($\text{SE} = \text{SD} / \sqrt{k}$).
   - Provide shared utility functions via `src/cv_utils.py` for all downstream notebooks.


## 1. Setup and Environment Initialization
*Code: Generated by AI Assistant*


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ensure local source modules can be imported
sys.path.append(os.path.abspath('..'))
from src.cv_utils import get_cv_splitter, back_transform_preds, evaluate_cv_model, RANDOM_STATE, N_SPLITS

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'

# Load raw dataset
data_path = os.path.join('..', 'data', 'hour.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'hour.csv')

df_raw = pd.read_csv(data_path)
print(f"Loaded raw dataset successfully. Shape: {df_raw.shape}")
df_raw.head()


Loaded raw dataset successfully. Shape: (17379, 17)


## 2. Data Quality Audit (Workflow Step 1)
*Code: Generated by AI Assistant*

Examine data quality strictly without analyzing predictor-outcome relationships to maintain unbiased split boundaries:
- Verification of rows, columns, and data types
- Missing-value markers (nulls, `NaN`, negative values)
- Duplicate records
- Integrity check: $\text{casual} + \text{registered} == \text{cnt}$


In [1]:
# 1. Dataset Dimensions & Column Types
print("--- Data Schema & Non-Null Counts ---")
df_raw.info()

# 2. Missing Value & Anomaly Scan
print("\n--- Missing Value Count per Feature ---")
missing_counts = df_raw.isnull().sum()
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "Zero missing values detected across all columns.")

# Check for placeholder string missing values in object columns
for col in df_raw.select_dtypes(include='object').columns:
    q_count = (df_raw[col] == '?').sum()
    if q_count > 0:
        print(f"Placeholder '?' detected in {col}: {q_count} rows")

# 3. Duplicate Rows
duplicates = df_raw.duplicated().sum()
print(f"\nDuplicate rows detected: {duplicates}")

# 4. Target Identity Verification (casual + registered == cnt)
component_sum = df_raw['casual'] + df_raw['registered']
discrepancies = (component_sum != df_raw['cnt']).sum()
print(f"Discrepancies in (casual + registered == cnt): {discrepancies}")
assert discrepancies == 0, 'Target variable cnt does not match sum of casual + registered!'


--- Data Schema & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB

--- Missin

In [1]:
# 5. Timestamp Continuity & Range
df_raw['dteday'] = pd.to_datetime(df_raw['dteday'])
date_min, date_max = df_raw['dteday'].min(), df_raw['dteday'].max()
total_days = (date_max - date_min).days + 1

print(f"Date range: {date_min.strftime('%Y-%m-%d')} to {date_max.strftime('%Y-%m-%d')} ({total_days} total calendar days)")
print(f"Total hours recorded: {len(df_raw):,} out of potential {total_days * 24:,} hours ({(len(df_raw)/(total_days * 24))*100:.2f}% coverage)")


Date range: 2011-01-01 to 2012-12-31 (731 total calendar days)
Total hours recorded: 17,379 out of potential 17,544 hours (99.06% coverage)


### Interpretation & Modeling Decisions: Data Quality Audit
**Author**: [Insert Name]  
<!-- Record your evaluation of data quality, missingness, duplicate checks, and target variable integrity here -->


## 3. Chronological Train / Test Split & Test Set Lock (Workflow Step 1)
*Code: Generated by AI Assistant*

Adjacent hours exhibit severe autocorrelation ($\rho_1 \approx 0.88, \rho_{24} \approx 0.70$). A random split would place neighboring hours in both train and test, producing optimistically leaking test errors.
- **Split Date**: `2012-10-01`
- **Training Set**: 2011-01-01 to 2012-09-30 (21 months, $N = 15,211$, 87.5%)
- **Holdout Test Set**: 2012-10-01 to 2012-12-31 (Q4 2012, $N = 2,168$, 12.5%)
- The holdout test set is locked into `data/test_locked.csv` and not called until Notebook 06.


In [1]:
# Execute chronological split
split_date = pd.to_datetime('2012-10-01')

train_mask = df_raw['dteday'] < split_date
test_mask = df_raw['dteday'] >= split_date

train_df = df_raw[train_mask].copy().reset_index(drop=True)
test_df = df_raw[test_mask].copy().reset_index(drop=True)

print(f"Training set: {len(train_df):,} rows ({len(train_df)/len(df_raw)*100:.1f}%) | Range: {train_df['dteday'].min().date()} to {train_df['dteday'].max().date()}")
print(f"Holdout test: {len(test_df):,} rows ({len(test_df)/len(df_raw)*100:.1f}%) | Range: {test_df['dteday'].min().date()} to {test_df['dteday'].max().date()}")

# Save split sets to disk
data_dir = os.path.join('..', 'data') if os.path.exists(os.path.join('..', 'data')) else 'data'
train_path = os.path.join(data_dir, 'train_clean.csv')
test_path = os.path.join(data_dir, 'test_locked.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"\nSaved training set to: {train_path}")
print(f"Saved locked test set to: {test_path} (Locked away for Notebook 06 audit)")


Training set: 15,211 rows (87.5%) | Range: 2011-01-01 to 2012-09-30
Holdout test: 2,168 rows (12.5%) | Range: 2012-10-01 to 2012-12-31

Saved training set to: ..\data\train_clean.csv
Saved locked test set to: ..\data\test_locked.csv (Locked away for Notebook 06 audit)


### Interpretation & Modeling Decisions: Chronological Split & Test Lock
**Author**: [Insert Name]  
<!-- Record your decisions on chronological partitioning, holdout proportion, and test set containment here -->


## 4. Feature Roles & Transformation Strategy (Workflow Step 3)
*Code: Generated by AI Assistant*

Cleanly distinguish between:
1. **Excluded Columns**:
   - `instant`, `dteday`: Index/date metadata redundant with calendar features.
   - `casual`, `registered`: Component targets dropped to prevent target leakage.
   - `atemp`: Feeling temperature dropped due to near-perfect collinearity with `temp` ($r = 0.9873$, $\text{VIF} > 300$).
2. **Data-Independent Transforms (Outside Pipeline)**:
   - Log target: $\log(cnt)$ (does not learn parameters from folds).
   - Temperature quadratic term: $\text{temp}^2$.
   - Category pooling: `weathersit = 4` pooled into `3` (sparsity remedy).
3. **Data-Dependent Transforms (Inside Pipeline)**:
   - `OneHotEncoder`: Learns discrete levels from training folds.
   - `StandardScaler`: Learns means and standard deviations from training folds.


In [1]:
# Apply data-independent transformations to training data
def apply_feature_engineering(df):
    data = df.copy()
    # 1. Quadratic temperature
    data['temp_sq'] = data['temp'] ** 2
    
    # 2. Pool rare weathersit category 4 (only 3 observations) into 3
    data['weathersit_pooled'] = data['weathersit'].replace({4: 3})
    
    # 3. Log target (where target is present)
    if 'cnt' in data.columns:
        data['log_cnt'] = np.log(data['cnt'])
        
    return data

train_df = apply_feature_engineering(train_df)
print(f"Engineered training features shape: {train_df.shape}")
print(f"weathersit_pooled distribution in training:\n{train_df['weathersit_pooled'].value_counts().sort_index()}")


Engineered training features shape: (15211, 20)
weathersit_pooled distribution in training:
weathersit_pooled
1    10189
2     3760
3     1262
Name: count, dtype: int64


In [1]:
# Define feature groupings for modeling
# 1. Features to drop entirely from modeling
DROP_COLS = ['instant', 'dteday', 'casual', 'registered', 'atemp']

# 2. Continuous numeric features
NUMERIC_FEATURES = ['temp', 'temp_sq', 'hum', 'windspeed']

# 3. Categorical features requiring dummy encoding in refined models
CATEGORICAL_FEATURES = ['hr', 'mnth', 'weathersit_pooled', 'weekday']

# 4. Binary / indicator features (0 or 1, pass through)
BINARY_FEATURES = ['yr', 'holiday', 'workingday']

print(f"Numeric features      : {NUMERIC_FEATURES}")
print(f"Categorical features  : {CATEGORICAL_FEATURES}")
print(f"Binary features       : {BINARY_FEATURES}")
print(f"Dropped features      : {DROP_COLS}")


Numeric features      : ['temp', 'temp_sq', 'hum', 'windspeed']
Categorical features  : ['hr', 'mnth', 'weathersit_pooled', 'weekday']
Binary features       : ['yr', 'holiday', 'workingday']
Dropped features      : ['instant', 'dteday', 'casual', 'registered', 'atemp']


### Interpretation & Modeling Decisions: Feature Roles & Transformation Strategy
**Author**: [Insert Name]  
<!-- Document rationale for dropped variables, pooling weathersit 4, and separating pipeline vs non-pipeline steps -->


## 5. Modular Pipeline Construction (Workflow Step 3)
*Code: Generated by AI Assistant*

Construct modular preprocessing pipelines using `ColumnTransformer` and `Pipeline`:
- **Baseline Pipeline**: Passes raw continuous/numeric features through for the naive model in Notebook 03.
- **Refined Pipeline**: Applies `OneHotEncoder(drop='first', sparse_output=False)` to categorical factors and `StandardScaler` to continuous features.
- Guarantees zero data leakage across CV folds (each fold estimates its own scaling means and encoder levels).


In [1]:
# Build scikit-learn ColumnTransformer objects

# 1. Baseline Preprocessor (Raw continuous features for naive baseline model)
baseline_cols = ['season', 'yr', 'mnth', 'hr', 'holiday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('pass', 'passthrough', baseline_cols)
    ],
    remainder='drop'
)

# 2. Refined Preprocessor (One-hot encoding for categorical factors + scaling for continuous features)
# Note: 'season' dropped to resolve extreme VIF with 'mnth'; 'weathersit_pooled' used instead of 'weathersit'
refined_categorical = ['hr', 'mnth', 'weathersit_pooled', 'weekday']
refined_numeric = ['temp', 'temp_sq', 'hum', 'windspeed']
refined_binary = ['yr', 'holiday', 'workingday']

refined_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), refined_categorical),
        ('num', StandardScaler(), refined_numeric),
        ('bin', 'passthrough', refined_binary)
    ],
    remainder='drop'
)

# Test fit on training data to verify output dimensions
X_tr_base = baseline_preprocessor.fit_transform(train_df)
X_tr_refined = refined_preprocessor.fit_transform(train_df)

print(f"Baseline feature matrix shape: {X_tr_base.shape} (10 raw features)")
print(f"Refined feature matrix shape : {X_tr_refined.shape} (Expanded dummy + scaled features)")


Baseline feature matrix shape: (15211, 10) (10 raw features)
Refined feature matrix shape : (15211, 49) (Expanded dummy + scaled features)


In [1]:
# Inspect generated feature names from refined preprocessor
cat_encoder = refined_preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(refined_categorical).tolist()
all_refined_names = cat_feature_names + refined_numeric + refined_binary

print(f"Total refined predictors: {len(all_refined_names)}")
print(f"First 10 feature names: {all_refined_names[:10]}")
print(f"Last 10 feature names : {all_refined_names[-10:]}")


Total refined predictors: 49
First 10 feature names: ['hr_1', 'hr_2', 'hr_3', 'hr_4', 'hr_5', 'hr_6', 'hr_7', 'hr_8', 'hr_9', 'hr_10']
Last 10 feature names : ['weekday_4', 'weekday_5', 'weekday_6', 'temp', 'temp_sq', 'hum', 'windspeed', 'yr', 'holiday', 'workingday']


### Interpretation & Modeling Decisions: Pipeline Architecture & Leakage Prevention
**Author**: [Insert Name]  
<!-- Discuss pipeline design, ColumnTransformer mechanics, and how fold-wise transformation prevents data leakage -->


## 6. Common Cross-Validation Protocol & Evaluation Metric (Workflow Step 4)
*Code: Generated by AI Assistant*

Following `docs/project_cv_workflow.html`:
- **Splitter**: `KFold(n_splits=10, shuffle=True, random_state=2026)`
- **Evaluation Metric**: Count-scale MSE and RMSE
- **Standard Error Formula**:
  $$\text{CV Error} = \frac{1}{k}\sum_{i=1}^k \text{MSE}_i, \quad \text{SE} = \frac{\text{SD of fold errors}}{\sqrt{k}}$$
- When evaluating log-transformed target models, back-transform fold predictions ($\exp(\hat{y} + \hat{\sigma}^2/2)$) before computing count-scale metrics.


In [1]:
# Verify shared CV splitter and evaluation function from src.cv_utils
cv_splitter = get_cv_splitter(n_splits=10, random_state=2026)
print(f"Shared CV Splitter: KFold(n_splits={cv_splitter.get_n_splits()}, shuffle=True, random_state={RANDOM_STATE})")

# Test evaluation function with a simple Pipeline
sample_pipeline = Pipeline([
    ('prep', baseline_preprocessor),
    ('reg', LinearRegression())
])

sample_results = evaluate_cv_model(
    estimator=sample_pipeline,
    X=train_df,
    y=train_df['cnt'],
    cv=cv_splitter,
    is_log_target=False
)

print("\n--- Test Run of Shared CV Evaluation Function ---")
print(f"Count-Scale Mean MSE  : {sample_results['mean_mse']:.2f}")
print(f"MSE Standard Error (SE): {sample_results['se_mse']:.2f}")
print(f"Count-Scale Mean RMSE : {sample_results['mean_rmse']:.2f}")
print(f"RMSE Standard Error    : {sample_results['se_rmse']:.2f}")
print(f"Count-Scale Mean MAE  : {sample_results['mean_mae']:.2f}")
print(f"Number of fold scores : {len(sample_results['fold_mse'])}")


Shared CV Splitter: KFold(n_splits=10, shuffle=True, random_state=2026)

--- Test Run of Shared CV Evaluation Function ---
Count-Scale Mean MSE  : 19016.28
MSE Standard Error (SE): 220.40
Count-Scale Mean RMSE : 137.88
RMSE Standard Error    : 0.80
Count-Scale Mean MAE  : 102.94
Number of fold scores : 10


### Interpretation & Modeling Decisions: CV Protocol & Metric Selection
**Author**: [Insert Name]  
<!-- Document choice of 10 folds, random_state=2026, count-scale MSE/RMSE metric, and standard error formula -->


## 7. Processed Artifacts Summary & Handoff to Notebook 03
*Code: Generated by AI Assistant*

Summary of exported artifacts for all downstream model notebooks:
- `data/train_clean.csv`: Training dataset ($N = 15,211$)
- `data/test_locked.csv`: Locked test holdout ($N = 2,168$)
- `src/cv_utils.py`: Reusable module with `get_cv_splitter()`, `evaluate_cv_model()`, and `back_transform_preds()`


In [1]:
# Audit exported artifacts on disk
print("=== Summary of Ready Artifacts ===")
artifacts = [
    (train_path, f"{len(train_df):,} rows"),
    (test_path, f"{len(test_df):,} rows (LOCKED)"),
    (os.path.join('..', 'src', 'cv_utils.py'), "Shared CV & evaluation module")
]

for p, desc in artifacts:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0
    status = "EXISTS" if exists else "MISSING"
    print(f"[{status}] {p} ({size_kb:.1f} KB) -> {desc}")


=== Summary of Ready Artifacts ===
[EXISTS] ..\data\train_clean.csv (989.9 KB) -> 15,211 rows
[EXISTS] ..\data\test_locked.csv (144.7 KB) -> 2,168 rows (LOCKED)
[EXISTS] ..\src\cv_utils.py (3.7 KB) -> Shared CV & evaluation module


### Summary & Handoff to Notebook 03 (Baseline Model)
**Author**: [Insert Name]  
<!-- Concluding remarks on data processing readiness and bridge to Notebook 03: Baseline Model & Failure Diagnosis -->
